# 09 — Initial **and** final steady states

**Ports**: `Code/MATLAB/Steady_State_Ayagari/calibrate_R_open_tg.m` and `solve_R.m`, plus `Main.m` lines 229–288.

**Why we need both**: Phase B calibrated $\beta^\star$ at the *Hungarian-data* NFA target ($\bar B/Y = -1.42$). But the transition experiment in the paper bridges two **different** steady states:
* the **initial** SS at $\text{BGss}_0 = 0$ (no net foreign assets), the starting point and the normalization for Figures 2–3;
* the **final** SS at $\text{BGss}_T = \lim_{t\to\infty} \text{shock\_b\_agg}(t) \approx -11.13$ (the long-run limit of the AR(1) credit-supply path, well below the calibrated $-4.34$). This SS provides `cpol_final` and `c_fine_final` to anchor the backward EGM along the transition.

Both are solved at the **same** $\beta = \beta^\star$ from notebook 07 (the deep parameter does not change across SSes — only foreign borrowing does).

**Strategy** (mirrors `calibrate_R_open_tg.m`):
1. Compute the export-demand-schedule constants $d_s$ and $\theta^\star$ from the calibrated SS (Main.m §II.c).
2. For a candidate $r$, derive $k/l$, $w$ from the firm FOC and solve a 1-D nonlinear equation for $l$ (labour FOC + budget).
3. Solve the household problem via EGM and form the stationary distribution.
4. Outer `brentq` over $r$ until $\text{BGss} + k - A^{HH}(r) = 0$.
5. Repeat for `BGss = 0` (initial) and `BGss = shock_b_agg[-1]` (final).

**Outputs (saved to `../output/initial_ss.npz`)**: aggregates and policy functions for both SSes plus $d_s$ and $\theta^\star$.

## Imports + load Phase B output

In [1]:
from pathlib import Path
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq

OUTPUT_DIR = Path('..') / 'output'
p = np.load(OUTPUT_DIR / 'params.npz')
cb = np.load(OUTPUT_DIR / 'calibration.npz')

Agrid = p['params__Agrid']
Agrid_fine = p['params__Agrid_fine']
piex = p['params__piex']
ex = p['params__ex']
exinv = p['params__exinv']
nA = int(p['params__nA'])
nA_fine = int(p['params__nA_fine'])
ns = int(p['params__ns'])
Index_b_min = int(p['params__Index_b_min'])
bmin = float(p['params__bmin'])
y_ss = float(p['params__y_ss'])
k_ss = float(p['params__k_ss'])
b_agg_ss = float(p['params__b_agg_ss'])
r_ss = float(p['params__r_ss'])
w_ss = float(p['params__w_ss'])
pi_ss = float(p['params__pi_ss'])
T_ss = float(p['params__T_ss'])
delta = float(p['params__delta'])
alpha = float(p['params__alpha'])
gamma = float(p['params__gamma'])
theta = float(p['params__theta'])
omega = float(p['params__omega'])
varphi = float(p['params__varphi'])
chi_dis = float(p['params__chi_dis'])
epsilon_f = float(p['params__epsilon_f'])
epsilon_w = float(p['params__epsilon_w'])
z_ss = float(p['params__z_ss'])
l_bar = float(p['params__l_bar'])

beta_calibrated = float(cb['beta_calibrated'])
G0_calibrated = cb['G0']
c_fine_calibrated = cb['c_fine']
dec_fine_calibrated = cb['dec_fine']
A_hh_calibrated = float(cb['A_hh'])

print(f'beta_calibrated  = {beta_calibrated:.6f}')
print(f'A_hh_calibrated  = {A_hh_calibrated:.6f}')
print(f'k_ss + b_agg_ss  = {k_ss + b_agg_ss:.6f}  (asset target at calibrated SS)')

beta_calibrated  = 0.983225
A_hh_calibrated  = 25.372094
k_ss + b_agg_ss  = 25.371939  (asset target at calibrated SS)


## Step 1 — derive $d_s$ and $\theta^\star$ from the calibrated SS

From `Main.m` §II.c (lines 229–254):
$$d_s \;=\; y_\text{ss} - C_H^{ss} - \delta k_\text{ss} \quad\text{(net export of the home good at the calibrated SS)}$$
$$\theta^\star \;=\; \left\lceil -\,(1+\tfrac{\omega}{1-\omega})^{-1}\,\bar B\,(1+r_\text{ss})/d_s\right\rceil$$
These two pin down the foreign export-demand schedule $C_H^\star = d_s\,T^{-\theta^\star}$ used everywhere in the rest of the paper.

In [2]:
ch_agg_ss_calibrated = (G0_calibrated * c_fine_calibrated).sum()
ch_star_agg = y_ss - ch_agg_ss_calibrated - k_ss * delta
d_s = ch_star_agg
mult_ss = 1.0 + T_ss ** (theta - 1.0) * omega / (1.0 - omega)
theta_star_raw = -(1.0 / mult_ss) * b_agg_ss * (1.0 + r_ss) / d_s
theta_star = int(np.ceil(theta_star_raw))

T_check = ((y_ss - ch_agg_ss_calibrated - k_ss * delta) / d_s) ** (-1.0 / theta_star)

print(f'C_H^ss        (sum c_fine * G0) = {ch_agg_ss_calibrated:.6f}')
print(f'C_H^star (export of home good)  = {ch_star_agg:.6f}')
print(f'd_s          (foreign demand)   = {d_s:.6f}')
print(f'theta_star raw                  = {theta_star_raw:.6f}')
print(f'theta_star ceil                 = {theta_star}')
print(f'T at calibrated SS              = {T_check:.6f} (should be 1.000000)')

C_H^ss        (sum c_fine * G0) = 1.453299
C_H^star (export of home good)  = 1.014945
d_s          (foreign demand)   = 1.014945
theta_star raw                  = 2.594818
theta_star ceil                 = 3
T at calibrated SS              = 1.000000 (should be 1.000000)


## Step 2 — inner block: prices, labour FOC, household problem

**Production block (firm FOC at SS)**: $r_K = r + \delta$, $k/l = (r_K\,\varepsilon_f / [\alpha(\varepsilon_f-1)])^{1/(\alpha-1)}$, $w = (k/l)^\alpha (1-\alpha)(\varepsilon_f-1)/\varepsilon_f$.

**Labour FOC** (calibrate_R_open_tg.m §find_l): set $C_h = (\varepsilon_w-1)/\varepsilon_w \cdot (1-\omega)\,w/\chi_\text{dis}\,l^{-\varphi}$ equal to total income net of foreign-good consumption. Solve for $l$ via `brentq`.

**Terms of trade**: $T = \big((y - C_h - \delta k)/d_s\big)^{-1/\theta^\star}$.

**Household problem (`solve_R.m`)**: standard EGM at fixed $(r, w, l, T, \Pi)$ using the calibrated $\beta^\star$.

In [3]:
def labor_block(r, BGss):
    """Mirror calibrate_R_open_tg.m lines 16-31."""
    r_K = r + delta
    k_l = (r_K / alpha * epsilon_f / (epsilon_f - 1.0)) ** (1.0 / (alpha - 1.0))
    w = k_l ** alpha * (1.0 - alpha) * (epsilon_f - 1.0) / epsilon_f

    def find_l_resid(x):
        rhs_ch = (epsilon_w - 1.0) / epsilon_w * (1.0 - omega) * w / chi_dis * x ** (-varphi)
        lhs_ch = (1.0 / mult_ss) * (
            r * (BGss + k_l * x)
            + z_ss * x * k_l ** alpha
              * ((epsilon_f - 1.0) / epsilon_f * (1.0 - alpha) + 1.0 / epsilon_f)
        )
        return rhs_ch - lhs_ch

    l = brentq(find_l_resid, 0.01, 10.0, xtol=1e-12)
    rhs_ch = (epsilon_w - 1.0) / epsilon_w * (1.0 - omega) * w / chi_dis * l ** (-varphi)
    c_h_agg = rhs_ch
    k = k_l * l
    y = z_ss * k ** alpha * l ** (1.0 - alpha)
    Profits = y / epsilon_f
    T = ((y - c_h_agg - k * delta) / d_s) ** (-1.0 / theta_star)
    return dict(k_l=k_l, w=w, l=l, k=k, y=y, c_h=c_h_agg, Profits=Profits, T=T)


def solve_egm_at(r, l, w, T, Profits, c0=None, beta=None, tol=1e-10, max_iter=5000):
    """Port of solve_R.m — EGM at given prices."""
    if beta is None:
        beta = beta_calibrated
    sav_grid = np.tile(Agrid[:, None], (1, ns))
    exLw = np.tile(ex[None, :], (nA, 1))
    mult = 1.0 + T ** (theta - 1.0) * omega / (1.0 - omega)
    if c0 is None:
        c0 = np.maximum(
            (1.0 / mult) * (w * l * exLw + r * sav_grid + Profits), 1e-5,
        )
    c_constrained = np.maximum(
        (1.0 / mult) * (w * l * exLw + (1.0 + r) * sav_grid + Profits - bmin), 1e-5,
    )
    c_new = np.empty_like(c0)
    a_today = np.empty_like(c0)
    c_s_last = np.empty_like(c0)
    for m in range(1, max_iter + 1):
        Emup = beta * (1.0 + r) * (c0 ** (-gamma) @ piex.T)
        c_s = Emup ** (-1.0 / gamma)
        a_today = (c_s * mult + sav_grid - w * l * exLw - Profits) / (1.0 + r)
        c_s_last = c_s
        for j in range(ns):
            xj = a_today[:, j]
            cj = c_s[:, j]
            threshold = a_today[Index_b_min, j]
            unconstrained_mask = Agrid > threshold
            interp = PchipInterpolator(xj, cj, extrapolate=True)
            c_new[:, j] = np.where(unconstrained_mask, interp(Agrid), c_constrained[:, j])
        c_new = np.maximum(c_new, 1e-5)
        max_diff = np.max(np.abs(c_new - c0))
        if max_diff < tol:
            break
        update = 0.5 if max_diff < 1e-4 else 0.9
        c0 = update * c_new + (1.0 - update) * c0
    cpol = c0
    dec = (1.0 + r) * Agrid[:, None] + exLw * w * l - cpol * mult + Profits
    c_fine = np.zeros((nA_fine, ns))
    dec_fine = np.zeros((nA_fine, ns))
    for j in range(ns):
        xj = a_today[:, j]
        cj = c_s_last[:, j]
        threshold = a_today[Index_b_min, j]
        mask_fine = Agrid_fine > threshold
        interp_unc = PchipInterpolator(xj, cj, extrapolate=True)
        interp_con = PchipInterpolator(Agrid, c_constrained[:, j], extrapolate=True)
        c_fine_j = np.where(mask_fine, interp_unc(Agrid_fine), interp_con(Agrid_fine))
        c_fine[:, j] = c_fine_j
        dec_unc = (1.0 + r) * Agrid_fine + ex[j] * w * l - c_fine_j * mult + Profits
        dec_fine[:, j] = np.where(mask_fine, dec_unc, bmin)
    return cpol, dec, c_fine, dec_fine, m, max_diff


def basefun_vec(grid, x):
    n = grid.size
    ind1 = np.searchsorted(grid, x, side='right') - 1
    ind1 = np.clip(ind1, 0, n - 2)
    ind2 = ind1 + 1
    w2 = (x - grid[ind1]) / (grid[ind2] - grid[ind1])
    w2 = np.clip(w2, 0.0, 1.0)
    w1 = 1.0 - w2
    return ind1, ind2, w1, w2


def stationary_distribution(dec_fine):
    ind1, ind2, w1, w2 = basefun_vec(Agrid_fine, dec_fine)
    i_idx = np.arange(nA_fine)
    j_idx = np.arange(ns)
    jp_idx = np.arange(ns)
    src = i_idx[:, None] + j_idx[None, :] * nA_fine
    src_b = src[:, :, None]
    dst1 = ind1[:, :, None] + jp_idx[None, None, :] * nA_fine
    dst2 = ind2[:, :, None] + jp_idx[None, None, :] * nA_fine
    val1 = piex[None, :, :] * w1[:, :, None]
    val2 = piex[None, :, :] * w2[:, :, None]
    rows = np.concatenate([
        np.broadcast_to(src_b, (nA_fine, ns, ns)).ravel(),
        np.broadcast_to(src_b, (nA_fine, ns, ns)).ravel(),
    ])
    cols = np.concatenate([dst1.ravel(), dst2.ravel()])
    vals = np.concatenate([val1.ravel(), val2.ravel()])
    N = nA_fine * ns
    transMat = sp.coo_matrix((vals, (rows, cols)), shape=(N, N)).tocsr()
    eig_vals, eig_vecs = eigs(transMat.T.tocsr(), k=1, which='LM', tol=1e-12, maxiter=10_000)
    EigVec = np.real(eig_vecs[:, 0])
    EigVec = EigVec / EigVec.sum()
    EigVec[EigVec < 0] = 0.0
    EigVec = EigVec / EigVec.sum()
    G0 = EigVec.reshape((ns, nA_fine)).T
    return G0


def calibrate_R_resid(r, BGss, c0=None):
    blk = labor_block(r, BGss)
    cpol, dec, c_fine, dec_fine, niter, mdiff = solve_egm_at(
        r, blk['l'], blk['w'], blk['T'], blk['Profits'], c0=c0,
    )
    G0 = stationary_distribution(dec_fine)
    A_hh = (G0 * dec_fine).sum()
    return BGss + blk['k'] - A_hh, blk, cpol, dec, c_fine, dec_fine, G0, A_hh, niter

## Sanity check: rerun the *calibrated* SS through this machinery

If we feed `BGss = b_agg_ss` and `r = r_ss`, the residual should be tiny (matches notebook 07's residual).

In [4]:
tic = time.time()
res_cal, blk_cal, *_ = calibrate_R_resid(r_ss, b_agg_ss)
print(f'  l = {blk_cal["l"]:.6f}, k = {blk_cal["k"]:.6f}, y = {blk_cal["y"]:.6f}')
print(f'  T = {blk_cal["T"]:.6f} (expect 1.0)')
print(f'  asset-market residual = {res_cal:+.4e} (expect tiny, matches NB07 residual)')
print(f'  elapsed: {time.time()-tic:.1f}s')

  l = 1.000000, k = 29.715189, y = 3.062548
  T = 1.000000 (expect 1.0)
  asset-market residual = -1.5474e-04 (expect tiny, matches NB07 residual)
  elapsed: 1.0s


## Step 3 — outer `brentq` over $r$ for the **initial** SS ($\text{BGss} = 0$)

When the country has **no** foreign borrowing, all of its capital must be financed domestically, so households need to hold less wealth in equilibrium. Lower $A^{HH}$ at the same $r$ means $r$ has to rise to clear the market. We thus expect $r_\text{initial} > r_\text{ss}$.

Sign of the residual $r(r) := k(r) - A^{HH}(r)$: at low $r$, $k(r)$ is large but agents save little, so residual is positive; at high $r$ residual is negative. From a quick scan (see comments below), the bracket is $r \in (0.010, 0.012)$.

In [5]:
def make_resid(BGss):
    cache = {'c0': None}
    def resid_for_r(r):
        res, *_rest = calibrate_R_resid(r, BGss, c0=cache['c0'])
        cpol = _rest[1]
        cache['c0'] = cpol.copy()
        print(f'  r = {r:.6f}: residual = {res:+.6f}')
        return res
    return resid_for_r

print('--- Initial SS (BGss = 0) ---')
tic = time.time()
r_initial = brentq(make_resid(0.0), 0.010, 0.012, xtol=1e-9)
print(f'r_initial = {r_initial:.8f}   (search took {time.time()-tic:.1f}s)')

res_init, blk_init, cpol_init, dec_init, c_fine_init, dec_fine_init, G0_init, A_hh_init, niter = (
    calibrate_R_resid(r_initial, 0.0)
)
print(f'Final EGM iterations    = {niter}')
print(f'Final asset-market resid= {res_init:+.4e}')

--- Initial SS (BGss = 0) ---


  r = 0.010000: residual = +7.734395


  r = 0.012000: residual = -5.461880


  r = 0.011172: residual = +0.727637


  r = 0.011270: residual = +0.065425


  r = 0.011279: residual = -0.000041


  r = 0.011279: residual = +0.000000


  r = 0.011279: residual = -0.000003
r_initial = 0.01127902   (search took 6.0s)


Final EGM iterations    = 1860
Final asset-market resid= +4.6422e-08


## Step 4 — final SS at the limit of the AR(1) credit-supply path

The AR(1) shock has a long-run limit `shock_b_agg[-1]`. The *final* SS is the steady state with this much foreign borrowing — different from the calibrated SS (which targets Hungarian data). The terminal policy functions `cpol_final, c_fine_final` from this SS are needed to anchor the backward EGM in `solve_transition_bis`.

Bracket: more foreign borrowing → more capital → lower MPK → lower r. So $r_\text{final} < r_\text{ss}$. We bracket below $r_\text{ss}$.

In [6]:
sh = np.load(OUTPUT_DIR / 'shock_path.npz')
BGss_final = float(sh['shock_b_agg'][-1])
print(f'BGss_final (limit of AR(1) shock) = {BGss_final:.6f}')
print()
print('--- Final SS (BGss = BGss_final) ---')
tic = time.time()
# Bracket: at lower r, k is huge; at calibrated r=0.0106, residual is still > 0 since
# more debt requires more saving. We test wide bounds.
r_final = brentq(make_resid(BGss_final), 0.001, 0.011, xtol=1e-9)
print(f'r_final = {r_final:.8f}   (search took {time.time()-tic:.1f}s)')

res_fin, blk_fin, cpol_fin, dec_fin, c_fine_fin, dec_fine_fin, G0_fin, A_hh_fin, niter_fin = (
    calibrate_R_resid(r_final, BGss_final)
)
print(f'Final EGM iterations    = {niter_fin}')
print(f'Final asset-market resid= {res_fin:+.4e}')

BGss_final (limit of AR(1) shock) = -11.132131

--- Final SS (BGss = BGss_final) ---


  r = 0.001000: residual = +36.549929


  r = 0.011000: residual = -9.203275


  r = 0.008988: residual = +1.997777


  r = 0.009347: residual = +0.250412


  r = 0.009397: residual = +0.000133


  r = 0.009397: residual = +0.000000


  r = 0.009397: residual = -0.000003
r_final = 0.00939734   (search took 5.3s)


Final EGM iterations    = 1518
Final asset-market resid= +5.4254e-09


## Step 5 — compare initial, calibrated, and final SS

In [7]:
rows = [
    ('r',           r_initial,            r_ss,                  r_final),
    ('r + delta',   r_initial + delta,    r_ss + delta,          r_final + delta),
    ('k/l',         blk_init['k_l'],      k_ss / 1.0,            blk_fin['k_l']),
    ('l',           blk_init['l'],        1.0,                   blk_fin['l']),
    ('k',           blk_init['k'],        k_ss,                  blk_fin['k']),
    ('y',           blk_init['y'],        y_ss,                  blk_fin['y']),
    ('w',           blk_init['w'],        w_ss,                  blk_fin['w']),
    ('C_h',         blk_init['c_h'],      ch_agg_ss_calibrated,  blk_fin['c_h']),
    ('Profits',     blk_init['Profits'],  pi_ss,                 blk_fin['Profits']),
    ('T',           blk_init['T'],        1.0,                   blk_fin['T']),
    ('A_hh',        A_hh_init,            A_hh_calibrated,       A_hh_fin),
    ('BGss',        0.0,                  b_agg_ss,              BGss_final),
    ('K + B',       blk_init['k'] + 0.0,  k_ss + b_agg_ss,       blk_fin['k'] + BGss_final),
]
print(f"{'quantity':<12} {'initial SS':>12} {'calibrated SS':>14} {'final SS':>10}")
print('-' * 52)
for name, init, cal, fin in rows:
    print(f'{name:<12} {init:>12.6f} {cal:>14.6f} {fin:>10.6f}')

quantity       initial SS  calibrated SS   final SS
----------------------------------------------------
r                0.011279       0.010610   0.009397
r + delta        0.031279       0.030610   0.029397
k/l             28.771348      29.715189  31.562890
l                0.984138       1.000000   1.022279
k               28.314968      29.715189  32.266065
y                2.982035       3.062548   3.193725
w                1.827150       1.846716   1.883847
C_h              1.449441       1.453299   1.466275
Profits          0.298204       0.306255   0.319373
T                1.016509       1.000000   0.978861
A_hh            28.314968      25.372094  21.133934
BGss             0.000000      -4.343250 -11.132131
K + B           28.314968      25.371939  21.133934


## Step 5 — save outputs

These will be loaded by notebooks 10 and 11 (transition + Figs 2–3).

In [8]:
out_path = OUTPUT_DIR / 'initial_ss.npz'
np.savez(
    out_path,
    # initial SS
    r_initial=r_initial, k_l_initial=blk_init['k_l'], l_initial=blk_init['l'],
    k_initial=blk_init['k'], y_initial=blk_init['y'], c_h_initial=blk_init['c_h'],
    w_initial=blk_init['w'], T_initial=blk_init['T'], Profits_initial=blk_init['Profits'],
    r_K_initial=r_initial + delta, BGss_initial=0.0, A_hh_initial=A_hh_init,
    G0_initial=G0_init, cpol_initial=cpol_init, dec_initial=dec_init,
    c_fine_initial=c_fine_init, dec_fine_initial=dec_fine_init,
    # final SS (terminal of the AR(1) credit-supply path)
    r_final=r_final, k_l_final=blk_fin['k_l'], l_final=blk_fin['l'],
    k_final=blk_fin['k'], y_final=blk_fin['y'], c_h_final=blk_fin['c_h'],
    w_final=blk_fin['w'], T_final=blk_fin['T'], Profits_final=blk_fin['Profits'],
    r_K_final=r_final + delta, BGss_final=BGss_final, A_hh_final=A_hh_fin,
    G0_final=G0_fin, cpol_final=cpol_fin, dec_final=dec_fin,
    c_fine_final=c_fine_fin, dec_fine_final=dec_fine_fin,
    # shared
    d_s=d_s, theta_star=theta_star,
)
print(f'Saved: {out_path.resolve()}')

Saved: /Users/siyingli/github/de-Ferra2020-kz/Code/Python/output/initial_ss.npz


### Notes on discrepancies (logged for `verification.md`)

- The **calibrated SS rerun** above produces the same residual ($\sim -1.5\times10^{-4}$) as notebook 07: this is the existing slack from `brentq` truncation in $\beta$, **not** new error from this notebook's prices solve.
- We use `brentq` instead of MATLAB's `fsolve` for the inner labour FOC (both are 1-D).
- The MATLAB script does *not* explicitly check that the calibrated SS is recovered from `calibrate_R_open_tg` — it *re-solves* via `fsolve` (Main.m line 262). We do the same recovery here as a sanity check before solving the initial SS.